# Hand check: HRDPS wind direction on the tilted grid

Phase 1, step 6. Before the extractor is written, this notebook does the whole job by hand for one site and one hour, so the numbers can be seen with our own eyes.

It reads the four HRDPS 10 m wind files for one hour (`UGRD`, `VGRD`, `WIND`, `WDIR`), finds the grid cell nearest Nickerson Bank, and shows three things:

1. Speed worked out from u and v matches ECCC's own `WIND` field.
2. Direction worked out from raw u and v, treating u as east, is about 46° wrong.
3. After turning u and v by the grid's local tilt, direction matches ECCC's own `WDIR` field.

`WIND` and `WDIR` are the answer key: ECCC computed them from the same u and v, with the rotation done properly.

Run every cell from the top (Run ▸ Run All Cells). The expected numbers are in comments at the end of each code cell.

In [1]:
import math
import numpy as np
import xarray as xr

# The site to check. Change these two numbers to try the other site
# (WSP Grand Banks: 47.24, -50.95) and re-run everything below.
SITE_LAT = 46.44
SITE_LON = -53.39

# One hour of one run: the 12Z run of 28 Oct 2026, 24 hours ahead.
# These four files are kept on the laptop; the VM has the live archive.
RUN = "20261028T12Z"
HOUR = "PT024H"

In [2]:
def open_hrdps(variable):
    """Open one HRDPS file and return its single data field."""
    path = f"../data/raw/hrdps/{RUN}_MSC_HRDPS_{variable}_AGL-10m_RLatLon0.0225_{HOUR}.grib2"
    # indexpath="" stops cfgrib writing .idx cache files next to the data
    ds = xr.open_dataset(path, engine="cfgrib", backend_kwargs={"indexpath": ""})
    # cfgrib names each field by its own convention: u10, v10, si10, wdir10
    return ds[list(ds.data_vars)[0]]

u = open_hrdps("UGRD")     # wind along the grid's rows
v = open_hrdps("VGRD")     # wind along the grid's columns
wind = open_hrdps("WIND")  # ECCC's own speed, m/s
wdir = open_hrdps("WDIR")  # ECCC's own direction, degrees, where the wind comes FROM
u

<xarray.DataArray 'u10' (y: 1290, x: 2540)> Size: 13MB
[3276600 values with dtype=float32]
Coordinates:
    latitude           (y, x) float64 26MB ...
    longitude          (y, x) float64 26MB ...
    time               datetime64[ns] 8B ...
    step               timedelta64[ns] 8B ...
    heightAboveGround  float64 8B ...
    valid_time         datetime64[ns] 8B ...
Dimensions without coordinates: y, x
Attributes: (12/33)
    GRIB_paramId:                             165
    GRIB_dataType:                            af
    GRIB_numberOfPoints:                      3276600
    GRIB_typeOfLevel:                         heightAboveGround
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_name:                                10 metre U wind component
    GRIB_shortName:                           10u
    GRIB_units:                               m s**-1
    long_name:                                10 metre U wind component
    units:                                    m s**-1
    standard_name:                            eastward_wind

## Step 1: find the cell nearest the site

Every grid point carries its own latitude and longitude, two tables of 1290 rows by 2540 columns. On this tilted grid a row is not a line of latitude, so the row can't be calculated from the site's latitude. Instead: measure the distance from the site to every point and keep the smallest.

The distance formula (haversine) follows the curve of the Earth, so that a degree of longitude (77 km here) and a degree of latitude (111 km) are measured in real kilometres.

In [3]:
lat = u.latitude.values   # shape (1290, 2540)
lon = u.longitude.values  # same shape; negative means west

def haversine_km(lat1, lon1, lat2, lon2):
    """Distance over the Earth's surface in km. Works on whole arrays at once."""
    earth_radius = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = p2 - p1
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2) ** 2
    return 2 * earth_radius * np.arcsin(np.sqrt(a))

dist = haversine_km(SITE_LAT, SITE_LON, lat, lon)  # one distance per grid point, 3.3 million of them
j, i = np.unravel_index(dist.argmin(), dist.shape)  # row and column of the smallest

print(f"nearest cell: row {j}, column {i}")
print(f"it sits at {lat[j, i]:.4f}N {-lon[j, i]:.4f}W, {dist[j, i]:.2f} km from the site")
# Expected for Nickerson Bank: row 955, column 2337, 1.14 km away

nearest cell: row 955, column 2337
it sits at 46.4436N 53.3761W, 1.14 km from the site


## Step 2: read the four values at that cell

In [4]:
u_val = float(u[j, i])
v_val = float(v[j, i])
eccc_speed = float(wind[j, i])
eccc_dir = float(wdir[j, i])

print(f"u = {u_val:.2f} m/s   v = {v_val:.2f} m/s")
print(f"ECCC says: speed {eccc_speed:.2f} m/s, direction {eccc_dir:.1f} degrees (from)")
# Expected: u 7.47, v 8.15; ECCC speed 11.07, direction 268.4 (a westerly)

u = 7.47 m/s   v = 8.15 m/s
ECCC says: speed 11.07 m/s, direction 268.4 degrees (from)


## Step 3: speed is right either way

u and v are the two sides of a right-angled triangle; the wind arrow is the long side. Its length doesn't depend on which way the triangle is turned, so speed comes out right even when the axes are wrong. That's why a speed check never catches this bug.

In [5]:
speed = math.hypot(u_val, v_val)  # sqrt(u^2 + v^2)
print(f"speed from u and v: {speed:.2f} m/s   (ECCC: {eccc_speed:.2f})")
# Expected: 11.06, a match (the last digit differs by rounding inside the file)

speed from u and v: 11.06 m/s   (ECCC: 11.07)


## Step 4: the wrong direction, treating u as east

`atan2(v, u)` gives the angle the arrow points *to*, counted anticlockwise from east, which is how maths does angles. Weather does it the other way round twice: compass bearings go clockwise from north, and wind direction is where the wind comes *from*, the opposite end of the arrow. `270 − angle` does both conversions in one step. (Check: an arrow pointing east, angle 0, gives 270, a wind from the west. Correct.)

In [6]:
def direction_from(u_east, v_north):
    """Compass direction the wind comes FROM, given true-east and true-north parts."""
    toward = math.degrees(math.atan2(v_north, u_east))  # anticlockwise from east
    return (270 - toward) % 360                          # clockwise from north, and "from"

naive_dir = direction_from(u_val, v_val)  # pretend u is east and v is north
print(f"direction if u were east: {naive_dir:.1f}   (ECCC: {eccc_dir:.1f})")
print(f"off by {abs(naive_dir - eccc_dir):.1f} degrees")
# Expected: 222.5, a south-westerly. Off by 45.9 degrees.

direction if u were east: 222.5   (ECCC: 268.4)
off by 45.9 degrees


## Step 5: measure how the row is tilted at this cell

u runs along the grid's row. So which real-world direction does the row run in here? Look at the next point along the same row (one column over) and ask: from our cell, which way does it lie?

A degree of longitude is shorter than a degree of latitude this far north, so the east–west step is scaled by cos(latitude) before the two are compared.

In [7]:
d_north = lat[j, i + 1] - lat[j, i]
d_east = (lon[j, i + 1] - lon[j, i]) * math.cos(math.radians(lat[j, i]))

tilt = math.degrees(math.atan2(d_north, d_east))  # angle of the row, anticlockwise from true east
side = "north" if tilt > 0 else "south"
print(f"the next point along the row lies {abs(tilt):.1f} degrees {side} of east")
# Expected at Nickerson Bank: 45.9 degrees south of east.
# So here "along the row" means roughly south-east, and "along the column" roughly north-east.

the next point along the row lies 45.9 degrees south of east


## Step 6: turn the arrow by the tilt, then compare

A 2-D rotation by the tilt angle re-expresses the same arrow in true-east and true-north parts. The arrow itself doesn't move, so its length stays 11.06.

In words, for this cell: 7.5 toward the south-east plus 8.2 toward the north-east. The south and north parts nearly cancel, and what's left is air moving east, which is a wind from the west.

In [8]:
a = math.radians(tilt)
u_east = u_val * math.cos(a) - v_val * math.sin(a)
v_north = u_val * math.sin(a) + v_val * math.cos(a)

rotated_dir = direction_from(u_east, v_north)
print(f"true-east part {u_east:.2f}, true-north part {v_north:.2f}")
print(f"direction after rotating: {rotated_dir:.1f}   (ECCC: {eccc_dir:.1f})")
print(f"speed after rotating: {math.hypot(u_east, v_north):.2f}   (unchanged)")
# Expected: 268.4, matching ECCC's WDIR. The east part is about 11, the north part about 0.

true-east part 11.05, true-north part 0.30
direction after rotating: 268.4   (ECCC: 268.4)
speed after rotating: 11.06   (unchanged)


## What to notice

- The error (45.9°) is exactly the local tilt of the grid. Near Vancouver the tilt is about 7°, so the same bug would be 7° there, and easy to miss.
- Nothing about the speed gave the bug away. Only comparing against `WDIR` did.
- The tilt here was measured from the file's own latitude and longitude tables. The textbook way computes it from the grid's pole position with trigonometry; at 2.5 km spacing the two agree to a fraction of a degree. The extractor will do the same measurement once per site and reuse it.

**Try it:** set `SITE_LAT`, `SITE_LON` to 47.24, −50.95 (WSP Grand Banks) in the first cell and run everything again. The cell, the tilt and the wind will all change; the rotated direction should still match `WDIR`.